# Polars in 20 minutes

In the spring you explored the Olist e-commerce orders with **pandas**. Here we ask a few of the same questions with **Polars**, to see how the two libraries differ.

Six moves cover most day-to-day work:

1. Read a file
2. Pick and add columns: `select`, `with_columns`
3. Group and count: `group_by`
4. Keep rows: `filter`
5. Chain the steps together
6. Combine tables: `join`

Data: two files from the [Brazilian E-Commerce Public Dataset (Kaggle)](https://www.kaggle.com/olistbr/brazilian-ecommerce), saved in this repo's `data/` folder.

This is the **tutorial** version: fill in the code cells. The complete version is `01_polars_wrangle.ipynb`.

In [ ]:
# import polars with its usual alias, pl
# --
import matplotlib.pyplot as plt

## 1. Read a file

| pandas | Polars |
|---|---|
| `pd.read_csv(path, parse_dates=[...])` | `pl.read_csv(path, try_parse_dates=True)` |

Notice the **data type under each column name**, and that there is **no index**: rows are just rows.

In [ ]:
# read ../data/olist_orders.csv.gz with try_parse_dates=True, then show its shape and first rows
orders = 


## 2. Pick and add columns: `select` and `with_columns`

Polars works with **expressions**: `pl.col("name")` means "the column called name", followed by what to do with it. Date functions live under `.dt`, text functions under `.str`.

| pandas | Polars |
|---|---|
| `df[["a", "b"]]` | `df.select("a", "b")` |
| `df["d"] = df["ts"].dt.date` | `df.with_columns(pl.col("ts").dt.date().alias("d"))` |

Polars never changes a DataFrame in place: each step returns a new one, so we assign the result back to `orders`.

In [ ]:
# keep four columns: order_id, customer_id, order_status, order_purchase_timestamp
orders = 

# add a column order_date: the date part of order_purchase_timestamp (hint: .dt.date() and .alias())
orders = 

orders.head()

## 3. Group and count: `group_by`

How many orders were placed each day?

| pandas | Polars |
|---|---|
| `df.groupby("d").size().reset_index()` | `df.group_by("d").len()` |

No index means no `reset_index()`: the group keys stay ordinary columns. Polars also does **not** sort groups for you, so add `.sort()` when order matters.

In [ ]:
# count orders per order_date, sorted by date
daily = 

daily.tail()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(daily["order_date"], daily["len"])
plt.ylabel("Number of orders")
plt.title("Daily orders");

## 4. Keep rows: `filter`

There's one huge spike. Which day was it?

| pandas | Polars |
|---|---|
| `df[df["n"] == df["n"].max()]` | `df.filter(pl.col("n") == pl.col("n").max())` |

In [ ]:
# keep the row where len equals the maximum len


## 5. Chain the steps

Each Polars method returns a new DataFrame, so you can chain the steps. Wrap the chain in parentheses to put one step per line, and read it top to bottom like a recipe.

What did orders look like, hour by hour, on the spike day?

In [ ]:
from datetime import date

# one chain: keep orders from 2017-11-24, add an "hour" column (.dt.hour()), count orders per hour, sort by hour
(
    orders
)

Busy all day, peaking late at night: not a glitch. 24 November 2017 was **Black Friday**.

## 6. Combine tables: `join`

`customer_id` in the orders file is **one ID per order**, not per person. The customers file maps it to `customer_unique_id`, the real customer.

| pandas | Polars |
|---|---|
| `df.merge(cust, on="customer_id", how="inner")` | `df.join(cust, on="customer_id", how="inner")` |

In [ ]:
# read ../data/olist_customers.csv.gz and keep customer_id and customer_unique_id
customers = 

# join it to orders on customer_id
orders = 

orders.head()

How many customers came back? Count orders per customer, then count customers per number of orders.

In [ ]:
# one chain: orders per customer (len(name="n_orders")), then customers per n_orders (len(name="n_customers")),
# sort by n_orders, and add a pct column: n_customers as a % of all customers, rounded to 1 decimal
(
    orders
)

About 97% of customers ordered only once. What would you do with that if you worked at Olist?

## Recap: pandas → Polars

| Task | pandas | Polars |
|---|---|---|
| Read | `pd.read_csv` | `pl.read_csv` (eager) or `pl.scan_csv` (lazy: next notebook) |
| Pick columns | `df[["a", "b"]]` | `df.select("a", "b")` |
| Add a column | `df["c"] = ...` | `df.with_columns(... .alias("c"))` |
| Keep rows | `df[mask]` | `df.filter(condition)` |
| Group and count | `df.groupby("a").size().reset_index()` | `df.group_by("a").len()` |
| Merge | `df.merge(other, on=...)` | `df.join(other, on=...)` |
| Hand off to scikit-learn | | `df.to_pandas()` or `df.to_numpy()` |